# 04 · Model comparison

Results of `experiments.py`: every model was trained under **identical conditions** (same training regions, seed, number of epochs, crops per epoch, loss, augmentation and learning-rate schedule). Only one factor changes per group:

| Group | Varied | Fixed |
|---|---|---|
| architecture | U-Net, U-Net++, DeepLabV3+, FPN, SegFormer | AdamW |
| encoder | ResNet-34, ResNet-50, EfficientNet-B4 | U-Net, AdamW |
| optimizer | AdamW, Adam, SGD with momentum | U-Net, ResNet-34 |

The baseline (U-Net, ResNet-34, AdamW) appears in every group as the reference.

**Model selection uses the validation regions only.** The winner is then evaluated once on the untouched test regions in notebook 03.

In [ ]:
import sys, json, shutil
from pathlib import Path

ROOT = Path.cwd() if Path("config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config as C

RES = ROOT / "results"
summ = []
hist = {}
for f in sorted(RES.glob("*/summary.json")):
    s = json.loads(f.read_text())
    summ.append(s)
    hist[s["name"]] = pd.read_csv(f.parent / "history.csv")
summary = pd.DataFrame(summ).sort_values("best_val_mIoU", ascending=False).reset_index(drop=True)
BASE = "unet_resnet34_adamw"
GROUP_COLORS = {"baseline": "black", "architecture": "tab:blue", "encoder": "tab:green", "optimizer": "tab:orange"}
print(f"{len(summary)} experiments, {summary['epochs'].iloc[0]} epochs each")

## 1 · Ranking by validation mIoU

In [ ]:
cols = ["name", "group", "arch", "encoder", "optimizer", "lr", "best_val_mIoU", "best_val_OA",
        "best_epoch", "params_M", "train_minutes"]
table = summary[cols].copy()
base_miou = summary.loc[summary.name == BASE, "best_val_mIoU"]
if len(base_miou):
    table["Δ vs baseline (pp)"] = 100 * (table["best_val_mIoU"] - base_miou.iloc[0])
table.style.format({"best_val_mIoU": "{:.3f}", "best_val_OA": "{:.3f}", "params_M": "{:.1f}",
                    "train_minutes": "{:.1f}", "lr": "{:.0e}", "Δ vs baseline (pp)": "{:+.1f}"}) \
     .background_gradient(subset=["best_val_mIoU"], cmap="Greens")

In [ ]:
# More stable comparison: average validation mIoU over the last 10 epochs (less luck than the single best epoch)
stable = pd.DataFrame({
    "name": list(hist),
    "val_mIoU_last10_mean": [h["val_mIoU"].tail(10).mean() for h in hist.values()],
    "val_mIoU_last10_std": [h["val_mIoU"].tail(10).std() for h in hist.values()],
}).merge(summary[["name", "group", "best_val_mIoU"]], on="name")
stable.sort_values("val_mIoU_last10_mean", ascending=False).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
order = summary.sort_values("best_val_mIoU")
ax.barh(order["name"], order["best_val_mIoU"], color=[GROUP_COLORS[g] for g in order["group"]])
for y, v in enumerate(order["best_val_mIoU"]):
    ax.text(v + 0.003, y, f"{v:.3f}", va="center", fontsize=9)
if len(base_miou):
    ax.axvline(base_miou.iloc[0], ls="--", c="grey", lw=1)
ax.set_xlabel("best validation mIoU"); ax.set_title("Model comparison")
lo = order["best_val_mIoU"].min()
ax.set_xlim(max(0, lo - 0.05), order["best_val_mIoU"].max() + 0.04)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]
ax.legend(handles, GROUP_COLORS.keys(), loc="lower right")
plt.tight_layout(); plt.savefig(RES / "comparison_bar.png", dpi=150); plt.show()

## 2 · Validation curves per group

In [ ]:
groups = ["architecture", "encoder", "optimizer"]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
for ax, g in zip(axes, groups):
    names = [BASE] + summary.loc[summary.group == g, "name"].tolist()
    for n in names:
        if n in hist:
            h = hist[n]
            ax.plot(h.epoch, h.val_mIoU, lw=2.5 if n == BASE else 1.8,
                    ls="--" if n == BASE else "-", label=n, color="black" if n == BASE else None)
    ax.set_title(f"Varying the {g}"); ax.set_xlabel("epoch"); ax.grid(alpha=.3); ax.legend(fontsize=8)
axes[0].set_ylabel("validation mIoU")
plt.tight_layout(); plt.savefig(RES / "comparison_curves.png", dpi=150); plt.show()

In [ ]:
# Training vs validation loss for every experiment: checks for overfitting or unstable training
n = len(hist)
ncol = 3
fig, axes = plt.subplots(int(np.ceil(n / ncol)), ncol, figsize=(16, 3.3 * np.ceil(n / ncol)), squeeze=False)
for ax, name in zip(axes.ravel(), summary["name"]):
    h = hist[name]
    ax.plot(h.epoch, h.train_loss, label="train"); ax.plot(h.epoch, h.val_loss, label="val")
    ax.set_title(name, fontsize=10); ax.grid(alpha=.3)
for ax in axes.ravel()[n:]:
    ax.axis("off")
axes[0, 0].legend()
plt.tight_layout(); plt.show()

## 3 · Per-class IoU

In [ ]:
iou_cols = [f"val_IoU_{c}" for c in C.CLASS_NAMES]
heat = summary.set_index("name")[iou_cols]
heat.columns = C.CLASS_NAMES
fig, ax = plt.subplots(figsize=(10, 0.55 * len(heat) + 1.5))
im = ax.imshow(heat.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v < .6 else "black", fontsize=9)
ax.set_xticks(range(len(C.CLASS_NAMES)), C.CLASS_NAMES, rotation=25, ha="right")
ax.set_yticks(range(len(heat)), heat.index)
ax.set_title("Validation IoU per class (best epoch of each model)")
plt.colorbar(im, fraction=.03); plt.tight_layout()
plt.savefig(RES / "comparison_per_class.png", dpi=150); plt.show()

## 4 · Accuracy vs cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, xcol, xlabel in [(axes[0], "params_M", "parameters (millions)"),
                         (axes[1], "train_minutes", "training time (minutes)")]:
    for _, r in summary.iterrows():
        ax.scatter(r[xcol], r["best_val_mIoU"], s=70, color=GROUP_COLORS[r["group"]], edgecolor="k")
        ax.annotate(r["name"], (r[xcol], r["best_val_mIoU"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
    ax.set_xlabel(xlabel); ax.set_ylabel("best validation mIoU"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5 · Select the final model
The model with the highest **validation** mIoU becomes `models/best_model.pth`, which notebook 03 and the app use. The checkpoint trained by notebook 02 is kept as a backup.

Differences of about 1 pp or less between models are within run-to-run noise (random crops, GPU non-determinism). In that case, prefer the smaller or faster model.

In [ ]:
# Chosen by the stable last-10-epoch average (tie with Adam; AdamW is steadier and is the baseline)
winner = summary[summary.name == "unet_resnet34_adamw"].iloc[0]
print(f"Winner: {winner['name']}  ({winner['arch']}, {winner['encoder']}, {winner['optimizer']})")
print(f"Validation mIoU {winner['best_val_mIoU']:.4f} at epoch {int(winner['best_epoch'])}")
if len(summary) > 1:
    gap = 100 * (winner["best_val_mIoU"] - summary.iloc[1]["best_val_mIoU"])
    print(f"Lead over 2nd place ({summary.iloc[1]['name']}): {gap:.2f} pp")

models_dir = ROOT / "models"; models_dir.mkdir(exist_ok=True)
target = models_dir / "best_model.pth"
backup = models_dir / "best_model_notebook02.pth"
if target.exists() and not backup.exists():
    shutil.copy2(target, backup)
    print("Backed up previous model to", backup.name)
shutil.copy2(RES / winner["name"] / "best.pth", target)
print("Copied winner to", target)

In [ ]:
# Validation confusion matrix of the winner (row-normalised = recall)
cm = np.array(json.loads((RES / winner["name"] / "summary.json").read_text())["best_confusion_matrix"], dtype=float)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
K = C.NUM_CLASSES
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(K):
    for j in range(K):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", color="white" if cmn[i, j] > .5 else "black")
ax.set_xticks(range(K), C.CLASS_NAMES, rotation=35, ha="right"); ax.set_yticks(range(K), C.CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("WorldCover"); ax.set_title(f"Validation confusion matrix · {winner['name']}")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

**Next:** run `03_evaluate.ipynb` to get the final test results of the selected model.